# Contextual Integrity — Hands-On
**ComplianceGPT Lab · REU 2026**

Companion to `slides-contextual-integrity.html`. No LLM calls needed for this one — everything here uses real, already-extracted data from `data/goldcoin_hhs_merged.csv`, so it runs instantly and identically for everyone.

**The idea:** Nissenbaum's Contextual Integrity 5-tuple (sender, receiver, subject, attribute, transmission principle) is literally this lab's extraction schema. You're going to tag scenarios with the tuple *by hand*, then check yourself against the real system, then find a real case where the tuple alone isn't enough to determine the verdict.

In [ ]:
import pandas as pd, json

df = pd.read_csv('/Users/priscilladanso/Documents/GitHub/COMPLIANCEGPT/data/goldcoin_hhs_merged.csv')
# The merged file has two source datasets (goldcoin, hhs) that reuse the same row_id
# numbering separately -- filter to goldcoin so row_id lookups below are unambiguous.
df = df[df['source'] == 'goldcoin'].reset_index(drop=True)
print(f'Loaded {len(df)} real GoldCoin scenarios (goldcoin source only, row_id is unique here)')
print(df.columns.tolist())

---
## Part 1 — Manual Tagging: You Be the Extractor

Below are 5 real scenario texts. For each, **before running any more code**, write down your own guess at the CI 5-tuple:
- **Sender** — who is disclosing / being asked to disclose
- **Receiver** — who receives it
- **Subject** — whose information it is
- **Attribute** — what type of information
- **Transmission principle** — the claimed reason/purpose the flow should be allowed

In [ ]:
practice_ids = [101, 70, 1, 20, 43]
practice_rows = df[df['row_id'].isin(practice_ids)]

for _, row in practice_rows.iterrows():
    print(f"--- row_id {row['row_id']} " + '-'*40)
    print(row['query'][:400])
    print('...\n')

**TODO — fill in your own guesses** (this is the actual exercise; don't skip to Part 2 without doing this):

In [ ]:
my_guesses = {
    101: {'sender': '...', 'receiver': '...', 'subject': '...', 'attribute': '...', 'transmission_principle': '...'},
    70:  {'sender': '...', 'receiver': '...', 'subject': '...', 'attribute': '...', 'transmission_principle': '...'},
    1:   {'sender': '...', 'receiver': '...', 'subject': '...', 'attribute': '...', 'transmission_principle': '...'},
    20:  {'sender': '...', 'receiver': '...', 'subject': '...', 'attribute': '...', 'transmission_principle': '...'},
    43:  {'sender': '...', 'receiver': '...', 'subject': '...', 'attribute': '...', 'transmission_principle': '...'},
}
print('Filled in — now run the next cell to check yourself.')

---
## Part 2 — Check Yourself Against the Real Extraction

In [ ]:
for _, row in practice_rows.iterrows():
    rid = row['row_id']
    sj = json.loads(row['scenario_json'])
    print(f"--- row_id {rid} (ground truth: {row['ground_truth']}) " + '-'*20)
    print(f"  sender:                {sj.get('sender')} ({sj.get('sender_role')})")
    print(f"  receiver:              {sj.get('receiver')} ({sj.get('receiver_role')})")
    print(f"  subject:               {sj.get('subject')} ({sj.get('subject_category')})")
    print(f"  attribute:             {sj.get('attribute')}")
    print(f"  transmission principle: {sj.get('purpose')}")
    print()

**Discuss with your table**: where did your guesses match, and where did they diverge? The most common mismatch is *sender_role vs. receiver_role* direction — did the record flow the way you thought, or backwards? That single confusion is one of the most common real extraction errors this lab has documented.

---
## Part 3 — C4 in the Wild: Identical Tuple, Opposite Verdict

This is not hypothetical. Rows 48 and 50 of this exact dataset share an **identical** CI 5-tuple — same sender role, receiver role, subject category, attribute, and transmission principle — but opposite ground-truth verdicts. Let's prove it and then find out why.

In [ ]:
def ci_tuple(scenario_json_str):
    d = json.loads(scenario_json_str)
    return (d.get('sender_role'), d.get('receiver_role'), d.get('subject_category'), d.get('attribute'), d.get('purpose'))

pair = df[df['row_id'].isin([48, 50])]
for _, row in pair.iterrows():
    print(f"row_id {row['row_id']} | verdict: {row['ground_truth']} | CI tuple: {ci_tuple(row['scenario_json'])}")

Confirmed — identical tuple, different verdict. Now let's see what actually distinguishes them. Read the full scenario text and the `extra_facts` field for each (these are the facts that don't fit into the core 5-tuple — exactly the C2/C4 territory from the slides).

In [ ]:
for _, row in pair.iterrows():
    print(f"=== row_id {row['row_id']} — {row['ground_truth']} " + '='*30)
    print(row['query'][:500])
    print()
    sj = json.loads(row['scenario_json'])
    print('extra_facts:')
    for f in sj.get('extra_facts', []):
        print(' ', f)
    print()

**TODO**: in one sentence, write down which specific fact (not in the 5-tuple) explains why row 48 is DENIED and row 50 is PERMITTED, even though their CI tuples are identical.

In [ ]:
my_explanation = "..."
print(my_explanation)

---
## Part 4 — Oracle Predicate Spotting

Given a new scenario, guess which oracle predicates (the C2 facts — court orders, authorizations, business associate agreements, etc.) are relevant *before* looking at the extraction.

In [ ]:
import random
random.seed(7)
spot_row = df[~df['row_id'].isin([48, 50] + practice_ids)].sample(1, random_state=7).iloc[0]
print(f"row_id {spot_row['row_id']}")
print(spot_row['query'][:500])

**TODO**: before running the next cell, write down: do you think a court order, patient authorization, business associate agreement, or some other enabling condition is present here? What in the text made you think so?

In [ ]:
my_oracle_guess = "..."

# Run this AFTER writing your guess above
sj = json.loads(spot_row['scenario_json'])
print('Actual extracted facts (non-empty only):')
for k, v in sj.items():
    if v not in (None, False, '', 'unspecified', []):
        print(f'  {k}: {v}')
print(f"\nGround truth verdict: {spot_row['ground_truth']}")

---
## Wrap-Up — Reframe Your Own Project

In 2–3 sentences: restate your project's research question using CI vocabulary (sender/receiver/subject/attribute/transmission principle, and — if relevant — which of C1–C4 your project is really about). If you're not sure, that's a good thing to bring to your next check-in.

In [ ]:
my_project_reframe = "..."
print(my_project_reframe)